In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="16ZxKwq4h36L8h8LJIFjH0rJ_fzC7M6Ks", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys
import numpy as np
import matplotlib.pyplot as plt

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime -> Change runtime type -> GPU")

print(f"\nPython {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set to {SEED}")

%matplotlib inline

# Attention Memory Scaling -- Vizuara

## 1. Why Does This Matter?

Modern language models want to process **long contexts** -- entire books, full codebases, hours of transcribed conversation. But the self-attention mechanism in a transformer creates an attention score matrix of shape $S \times S$, where $S$ is the sequence length. This means attention memory grows **quadratically** with sequence length.

In this notebook, we will:
1. **Measure** the actual GPU memory consumed by attention at different sequence lengths
2. **Verify** the $O(S^2)$ scaling law empirically
3. **Visualize** the quadratic wall and identify where single-GPU attention becomes impossible
4. **Implement** chunked attention (the seed idea behind Ring Attention) and verify it produces identical results

By the end, you will have hands-on evidence for why context parallelism is necessary, and you will understand the online softmax trick that makes chunked attention numerically exact.

In [ ]:
#@title 🎧 Listen: Why It Matters Attention Matrix
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_why_it_matters_attention_matrix.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. The Attention Matrix: How Big Is It?

In standard self-attention, for a single head with head dimension $d_h$ and sequence length $S$:

$$\mathbf{A} = \mathbf{Q} \mathbf{K}^T \quad \text{shape: } S \times S$$

In FP16 (2 bytes per element), the memory for this matrix is $2 S^2$ bytes. Let us compute this for various sequence lengths.

In [ ]:
#@title 🎧 Code Walkthrough: Theoretical Sizes
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_theoretical_sizes.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Compute theoretical attention matrix sizes
seq_lengths = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]

print(f"{'Seq Length':>12} | {'Matrix Shape':>22} | {'FP16 Memory':>15} | {'FP32 Memory':>15}")
print("-" * 75)

for S in seq_lengths:
    entries = S * S
    fp16_bytes = entries * 2
    fp32_bytes = entries * 4

    def format_bytes(b):
        if b < 1e6:
            return f"{b / 1e3:.0f} KB"
        elif b < 1e9:
            return f"{b / 1e6:.0f} MB"
        elif b < 1e12:
            return f"{b / 1e9:.1f} GB"
        else:
            return f"{b / 1e12:.1f} TB"

    print(f"{S:>12,} | {S:>10,} x {S:<10,} | {format_bytes(fp16_bytes):>15} | {format_bytes(fp32_bytes):>15}")

print("\nKey insight: Memory quadruples every time sequence length doubles.")
print("At 128K tokens, a SINGLE HEAD requires 32 GB in FP16.")
print("With 32 heads, that is 1 TB per layer -- far beyond any GPU.")

In [ ]:
#@title 🎧 Code Walkthrough: Measuring Real Gpu Memory
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_measuring_real_gpu_memory.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Measuring Real GPU Memory

Let us go beyond theory and actually **measure** how much GPU memory PyTorch uses for attention at different sequence lengths. We will allocate Q, K, V tensors and compute attention, tracking peak memory.

In [ ]:
def measure_attention_memory(seq_len, d_head=64, dtype=torch.float16):
    """
    Measure the peak GPU memory used by standard attention for a given sequence length.
    Uses a single attention head, batch size 1.
    """
    if not torch.cuda.is_available():
        return None

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Record baseline memory
    baseline = torch.cuda.memory_allocated()

    try:
        # Create Q, K, V: (1, seq_len, d_head)
        Q = torch.randn(1, seq_len, d_head, device='cuda', dtype=dtype)
        K = torch.randn(1, seq_len, d_head, device='cuda', dtype=dtype)
        V = torch.randn(1, seq_len, d_head, device='cuda', dtype=dtype)

        # Compute attention scores: S x S matrix
        scores = torch.bmm(Q, K.transpose(-2, -1)) / (d_head ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        output = torch.bmm(attn_weights, V)

        # Peak memory used
        peak = torch.cuda.max_memory_allocated()
        used = peak - baseline

        # Clean up
        del Q, K, V, scores, attn_weights, output
        torch.cuda.empty_cache()

        return used

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return -1  # OOM marker
        raise


# Measure for increasing sequence lengths
# We stay conservative to avoid OOM on T4 (16 GB)
test_lengths = [512, 1024, 2048, 4096, 8192, 12288, 16384]
d_head = 64

measured_memory = []
theoretical_memory = []

print(f"Measuring attention memory (d_head={d_head}, FP16, single head, batch=1)")
print(f"{'Seq Length':>10} | {'Measured':>12} | {'Theoretical (S^2 * 2B)':>22} | {'Ratio':>8}")
print("-" * 65)

for S in test_lengths:
    mem = measure_attention_memory(S, d_head=d_head)
    if mem is None:
        print(f"{S:>10,} | {'No GPU':>12} | {'---':>22} | {'---':>8}")
        continue
    if mem == -1:
        print(f"{S:>10,} | {'OOM':>12} | {'---':>22} | {'---':>8}")
        break

    # Theoretical: Q, K, V tensors + attention matrix + output
    # QKV: 3 * S * d_head * 2 bytes
    # Attn matrix: S * S * 2 bytes (FP16) or S * S * 4 (FP32 after softmax)
    # Output: S * d_head * 2 bytes
    theory_attn = S * S * 4  # softmax often computed in FP32
    theory_qkv = 3 * S * d_head * 2
    theory_out = S * d_head * 2
    theory_total = theory_attn + theory_qkv + theory_out

    measured_memory.append(mem)
    theoretical_memory.append(theory_total)

    def fmt(b):
        if b < 1e6: return f"{b/1e3:.0f} KB"
        elif b < 1e9: return f"{b/1e6:.1f} MB"
        else: return f"{b/1e9:.2f} GB"

    ratio = mem / theory_total if theory_total > 0 else 0
    print(f"{S:>10,} | {fmt(mem):>12} | {fmt(theory_total):>22} | {ratio:>8.2f}x")

print("\nThe 'Measured' column is actual GPU memory from PyTorch.")
print("The ratio may exceed 1.0 due to PyTorch memory allocator overhead.")

In [ ]:
#@title 🎧 What to Look For: Verifying Scaling Law
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_verifying_scaling_law.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Verifying the Quadratic Scaling Law

If memory truly scales as $O(S^2)$, then when we double the sequence length, memory should roughly **quadruple**. Let us plot the measured data and fit the scaling exponent.

In [ ]:
if len(measured_memory) >= 3:
    # Compute scaling ratios between consecutive measurements
    print("Scaling Verification:")
    print(f"{'S1 -> S2':>16} | {'Seq Ratio':>10} | {'Memory Ratio':>13} | {'Expected (R^2)':>14}")
    print("-" * 60)

    valid_lengths = test_lengths[:len(measured_memory)]

    for i in range(1, len(measured_memory)):
        seq_ratio = valid_lengths[i] / valid_lengths[i-1]
        mem_ratio = measured_memory[i] / measured_memory[i-1] if measured_memory[i-1] > 0 else 0
        expected_ratio = seq_ratio ** 2
        print(f"{valid_lengths[i-1]:>6} -> {valid_lengths[i]:<6} | {seq_ratio:>10.1f}x | {mem_ratio:>12.2f}x | {expected_ratio:>13.1f}x")

    # Fit log-log line to extract scaling exponent
    log_S = np.log2(np.array(valid_lengths, dtype=float))
    log_mem = np.log2(np.array(measured_memory, dtype=float))
    coeffs = np.polyfit(log_S, log_mem, 1)
    print(f"\nFitted scaling exponent: {coeffs[0]:.2f}")
    print(f"Expected for O(S^2): 2.00")
    print(f"The measured exponent should be close to 2.0, confirming quadratic scaling.")

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Linear scale
    ax1.plot(valid_lengths, np.array(measured_memory) / 1e6, 'bo-', markersize=8, label='Measured')
    ax1.plot(valid_lengths, np.array(theoretical_memory) / 1e6, 'r--', alpha=0.7, label='Theoretical')
    ax1.set_xlabel('Sequence Length', fontsize=12)
    ax1.set_ylabel('Memory (MB)', fontsize=12)
    ax1.set_title('Attention Memory vs Sequence Length\n(Linear Scale)', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # Log-log scale
    ax2.loglog(valid_lengths, np.array(measured_memory) / 1e6, 'bo-', markersize=8, label='Measured')
    ax2.loglog(valid_lengths, np.array(theoretical_memory) / 1e6, 'r--', alpha=0.7, label='Theoretical')
    # Fit line
    fit_x = np.linspace(log_S[0], log_S[-1], 100)
    fit_y = coeffs[0] * fit_x + coeffs[1]
    ax2.loglog(2**fit_x, 2**fit_y / 1e6, 'g-', alpha=0.5, linewidth=2,
               label=f'Fit: slope = {coeffs[0]:.2f}')
    ax2.set_xlabel('Sequence Length', fontsize=12)
    ax2.set_ylabel('Memory (MB)', fontsize=12)
    ax2.set_title('Attention Memory vs Sequence Length\n(Log-Log Scale -- slope ~ 2 = quadratic)', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3, which='both')

    plt.tight_layout()
    plt.show()
else:
    print("Not enough data points for scaling analysis. Need GPU with sufficient memory.")

In [ ]:
#@title 🎧 What to Look For: Quadratic Wall Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_quadratic_wall_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. The Quadratic Wall: When Does a Single GPU Fail?

Let us extrapolate our measurements to predict at what sequence length a single GPU runs out of memory. We will project for different GPU types (T4 16 GB, A100 40 GB, A100 80 GB, H100 80 GB).

In [ ]:
# Extrapolate attention memory to large sequence lengths
# Using the theoretical formula: mem ≈ S^2 * 4 bytes (FP32 attn) + S * d_head * 8 (QKV+O)
d_head = 128  # typical head dimension in modern models
num_heads = 32  # typical number of heads

seq_range = np.array([4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576])

# Per-head attention memory (dominant term: S^2 * bytes_per_element)
# With FlashAttention off: full S x S matrix materialized
attn_mem_per_head = seq_range.astype(float) ** 2 * 2  # FP16
attn_mem_total = attn_mem_per_head * num_heads

# GPU memory limits
gpu_limits = {
    'T4 (16 GB)': 16e9,
    'A100-40GB': 40e9,
    'A100-80GB': 80e9,
    'H100-80GB': 80e9,
}

fig, ax = plt.subplots(figsize=(12, 6))

# Plot attention memory curve
ax.semilogy(seq_range, attn_mem_total / 1e9, 'b-o', markersize=6, linewidth=2,
            label=f'Attention Memory ({num_heads} heads, FP16)')

# Plot GPU limits
colors = ['#e74c3c', '#e67e22', '#f39c12', '#2ecc71']
for (name, limit), color in zip(gpu_limits.items(), colors):
    ax.axhline(y=limit / 1e9, color=color, linestyle='--', linewidth=1.5,
               label=f'{name}', alpha=0.8)

ax.set_xlabel('Sequence Length (tokens)', fontsize=13)
ax.set_ylabel('Memory (GB) -- log scale', fontsize=13)
ax.set_title(f'The Quadratic Wall: Attention Memory for {num_heads} Heads (d_h={d_head})',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3, which='both')

# Annotate crossover points
for name, limit in gpu_limits.items():
    # Find where attn memory exceeds GPU limit
    exceed_idx = np.where(attn_mem_total > limit)[0]
    if len(exceed_idx) > 0:
        idx = exceed_idx[0]
        ax.annotate(f'{name} limit exceeded\nat S={seq_range[idx]:,}',
                    xy=(seq_range[idx], attn_mem_total[idx] / 1e9),
                    xytext=(seq_range[idx] * 2, attn_mem_total[idx] / 1e9 * 3),
                    fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'),
                    color='gray')

ax.set_xticks(seq_range)
ax.set_xticklabels([f'{s//1024}K' if s < 1048576 else '1M' for s in seq_range],
                   rotation=45, fontsize=9)

plt.tight_layout()
plt.show()

print("\nConclusion: For 32 heads with d_h=128, attention memory exceeds 80 GB")
print("well before 64K tokens. Long-context training REQUIRES distributing attention.")

In [ ]:
#@title 🎧 Listen: Online Softmax Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_online_softmax_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Online Softmax: The Mathematical Key to Chunked Attention

The standard softmax requires seeing all $S$ values before computing the normalization:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{S} e^{x_j}}$$

But the **online softmax** trick lets us accumulate the result incrementally. We maintain:
- $m$: the running maximum of all logits seen so far
- $l$: the running sum of exponentiated (re-centered) logits
- $o$: the running weighted sum of values

When a new chunk of logits arrives, we update these accumulators. After processing all chunks, we get the exact same result as full softmax.

Let us implement this and verify correctness.

In [ ]:
#@title 🎧 Code Walkthrough: Online Softmax Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_online_softmax_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def standard_attention(Q, K, V):
    """
    Standard full-matrix attention.
    Q, K, V: (batch, seq_len, d_head)
    Returns: (batch, seq_len, d_head)
    """
    d_h = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_h ** 0.5)  # (B, S, S)
    attn_weights = torch.softmax(scores, dim=-1)       # (B, S, S)
    output = attn_weights @ V                          # (B, S, d_h)
    return output


def chunked_attention_online_softmax(Q, K, V, num_chunks=4):
    """
    Chunked attention using online softmax.
    Splits K, V into chunks and processes them one at a time.
    Produces EXACTLY the same result as standard attention.
    """
    B, S, d_h = Q.shape
    chunk_size = S // num_chunks

    # Initialize accumulators
    running_max = torch.full((B, S, 1), float('-inf'), device=Q.device, dtype=Q.dtype)
    running_sum = torch.zeros((B, S, 1), device=Q.device, dtype=Q.dtype)
    running_out = torch.zeros((B, S, d_h), device=Q.device, dtype=Q.dtype)

    for c in range(num_chunks):
        start = c * chunk_size
        end = start + chunk_size

        K_chunk = K[:, start:end, :]  # (B, chunk_size, d_h)
        V_chunk = V[:, start:end, :]  # (B, chunk_size, d_h)

        # Compute logits for this chunk: (B, S, chunk_size)
        logits = Q @ K_chunk.transpose(-2, -1) / (d_h ** 0.5)

        # Online softmax update
        chunk_max = logits.max(dim=-1, keepdim=True).values  # (B, S, 1)
        new_max = torch.maximum(running_max, chunk_max)

        # Rescale old accumulators to new max
        alpha = torch.exp(running_max - new_max)
        # Exponentiate new logits under new max
        beta = torch.exp(logits - new_max)

        # Update
        running_sum = alpha * running_sum + beta.sum(dim=-1, keepdim=True)
        running_out = alpha * running_out + beta @ V_chunk
        running_max = new_max

    # Final normalization
    output = running_out / running_sum
    return output


# Test: Compare standard vs chunked attention
B, S, d_h = 1, 256, 64
Q = torch.randn(B, S, d_h)
K = torch.randn(B, S, d_h)
V = torch.randn(B, S, d_h)

out_standard = standard_attention(Q, K, V)
out_chunked = chunked_attention_online_softmax(Q, K, V, num_chunks=4)

diff = (out_standard - out_chunked).abs().max().item()
print(f"Max absolute difference: {diff:.2e}")
print(f"Results are {'IDENTICAL (within floating point)' if diff < 1e-5 else 'DIFFERENT -- bug!'}")
print(f"\nStandard attention peak memory: O(S^2) = O({S}^2) = {S*S} entries")
print(f"Chunked attention peak memory: O(S * S/4) = O({S} * {S//4}) = {S * (S//4)} entries")
print(f"Memory reduction: {S*S / (S * (S//4)):.1f}x")

In [ ]:
#@title 🎧 Narration: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_todo_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Your Turn

### Exercise 1: Verify Chunked Attention with Different Chunk Counts

The online softmax trick should give **exactly the same output** regardless of how many chunks we split K, V into. Verify this by testing with 1 chunk (which is standard attention), 2, 4, 8, and 16 chunks.

In [ ]:
#@title 🎧 Before You Start: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_todo_1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Verify that chunked attention gives the same output for different chunk counts

B, S, d_h = 1, 512, 64
Q = torch.randn(B, S, d_h)
K = torch.randn(B, S, d_h)
V = torch.randn(B, S, d_h)

reference = standard_attention(Q, K, V)

chunk_counts = [1, 2, 4, 8, 16]

for nc in chunk_counts:
    # TODO: Call chunked_attention_online_softmax with num_chunks=nc
    # TODO: Compute the max absolute difference from the reference
    # TODO: Print the result
    out = None  # REPLACE THIS LINE
    diff = 0.0  # REPLACE THIS LINE
    print(f"  Chunks={nc:>2}: max diff = {diff:.2e}")

print("\nAll differences should be < 1e-5 (floating point noise).")

In [ ]:
#@title 🎧 Listen: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 2: Measure Memory Savings from Chunking

When we use chunked attention with $C$ chunks, the largest intermediate tensor changes from $S \times S$ to $S \times (S/C)$. Measure the actual GPU memory difference.

In [ ]:
#@title 🎧 Before You Start: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_todo_2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Measure GPU memory for standard vs chunked attention

def measure_chunked_attention_memory(seq_len, num_chunks, d_head=64, dtype=torch.float16):
    """
    Measure the peak GPU memory for chunked attention.
    """
    if not torch.cuda.is_available():
        return None

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline = torch.cuda.memory_allocated()

    try:
        Q = torch.randn(1, seq_len, d_head, device='cuda', dtype=dtype)
        K = torch.randn(1, seq_len, d_head, device='cuda', dtype=dtype)
        V = torch.randn(1, seq_len, d_head, device='cuda', dtype=dtype)

        # TODO: Call chunked_attention_online_softmax with the specified num_chunks
        # Make sure to move Q, K, V to GPU first!
        output = None  # REPLACE THIS LINE

        peak = torch.cuda.max_memory_allocated()
        used = peak - baseline

        del Q, K, V, output
        torch.cuda.empty_cache()
        return used

    except RuntimeError:
        torch.cuda.empty_cache()
        return -1


# TODO: Compare memory for S=8192 with 1 chunk (full attention) vs 2, 4, 8 chunks
# Hint: measure_attention_memory(8192) gives the baseline
# Then measure_chunked_attention_memory(8192, num_chunks=N) for each N

S = 8192
print(f"Memory comparison for S={S}, d_head=64, FP16:")
print(f"{'Chunks':>8} | {'Memory':>12} | {'Savings':>10}")
print("-" * 40)

# REPLACE: fill in the measurement loop
pass

In [ ]:
#@title 🎧 Listen: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_todo_3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 3: Online Softmax from Scratch

Implement the online softmax update step yourself. Given partial logits from two chunks, combine them to get the exact softmax result.

In [ ]:
#@title 🎧 Before You Start: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_todo_3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Implement online softmax for a single query attending to two KV chunks

# Setup: a single query token, two KV chunks of 4 tokens each
torch.manual_seed(42)
d_h = 4
q = torch.randn(d_h)           # single query
k1 = torch.randn(4, d_h)       # KV chunk 1 (4 keys)
v1 = torch.randn(4, d_h)       # KV chunk 1 (4 values)
k2 = torch.randn(4, d_h)       # KV chunk 2 (4 keys)
v2 = torch.randn(4, d_h)       # KV chunk 2 (4 values)

# Ground truth: full attention over all 8 KV tokens
K_full = torch.cat([k1, k2], dim=0)   # (8, d_h)
V_full = torch.cat([v1, v2], dim=0)   # (8, d_h)
logits_full = q @ K_full.T / (d_h ** 0.5)  # (8,)
weights_full = torch.softmax(logits_full, dim=0)  # (8,)
output_full = weights_full @ V_full  # (d_h,)

# --- YOUR IMPLEMENTATION: Online softmax in two steps ---

# Step 1: Process chunk 1
logits1 = q @ k1.T / (d_h ** 0.5)  # (4,)
m1 = None  # TODO: max of logits1
l1 = None  # TODO: sum of exp(logits1 - m1)
o1 = None  # TODO: exp(logits1 - m1) @ v1

# Step 2: Process chunk 2 and combine with chunk 1
logits2 = q @ k2.T / (d_h ** 0.5)  # (4,)
m2 = None  # TODO: max of logits2
new_max = None  # TODO: max(m1, m2)
alpha = None  # TODO: exp(m1 - new_max)
beta = None  # TODO: exp(logits2 - new_max)
new_l = None  # TODO: alpha * l1 + sum(beta)
new_o = None  # TODO: alpha * o1 + beta @ v2

# Final output
output_online = None  # TODO: new_o / new_l

# Verify
if output_online is not None:
    diff = (output_full - output_online).abs().max().item()
    print(f"Full attention output:   {output_full.numpy()}")
    print(f"Online softmax output:   {output_online.numpy()}")
    print(f"Max difference:          {diff:.2e}")
    print(f"Match: {'YES' if diff < 1e-5 else 'NO -- check your implementation'}")
else:
    print("Fill in the TODO sections above to complete this exercise.")

In [ ]:
#@title 🎧 What to Look For: Connecting To Context Parallelism
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_connecting_to_context_parallelism.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Connecting to Context Parallelism

The chunked attention we just implemented is the **single-GPU version** of what Ring Attention does across multiple GPUs:

| Single-GPU Chunked | Multi-GPU Ring Attention |
|---|---|
| Split K, V into chunks in memory | Split K, V across GPUs |
| Loop over chunks sequentially | Rotate KV chunks around the ring |
| Online softmax combines partial results | Same online softmax on each GPU |
| Memory: $O(S \cdot S/C)$ | Memory per GPU: $O((S/N)^2)$ |

The online softmax trick is exactly the same. The only difference is that in Ring Attention, the KV chunks arrive via network transfer instead of being read from local memory.

In [ ]:
# Visualize how context parallelism reduces per-GPU memory
S_values = [32768, 65536, 131072, 262144, 524288, 1048576]
N_gpus = [1, 2, 4, 8, 16, 32]

fig, ax = plt.subplots(figsize=(12, 6))

for N in [1, 4, 8, 16]:
    # Per-GPU attention memory: (S/N)^2 * 2 bytes * num_heads
    per_gpu_mem = [(s / N) ** 2 * 2 * 32 / 1e9 for s in S_values]
    label = f'N={N} GPU{"s" if N > 1 else ""}'
    if N == 1:
        label = 'No parallelism'
    ax.semilogy(range(len(S_values)), per_gpu_mem, 'o-', markersize=7, linewidth=2, label=label)

# GPU memory lines
ax.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='A100 80GB')
ax.axhline(y=16, color='orange', linestyle='--', alpha=0.7, label='T4 16GB')

ax.set_xticks(range(len(S_values)))
ax.set_xticklabels([f'{s//1024}K' if s < 1048576 else '1M' for s in S_values], fontsize=10)
ax.set_xlabel('Sequence Length', fontsize=13)
ax.set_ylabel('Per-GPU Attention Memory (GB) -- log scale', fontsize=13)
ax.set_title('Context Parallelism: Per-GPU Memory Reduction\n(32 heads, FP16, per-chunk attention)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("With 8-way context parallelism, 128K-token attention fits in ~512 MB per GPU.")
print("Without it, the same attention requires ~32 GB per head (1 TB for 32 heads).")
print("Context parallelism gives a QUADRATIC reduction in per-GPU memory: O(S^2/N^2).")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Summary

In this notebook we verified the key facts that motivate context parallelism:

1. **Attention memory scales as $O(S^2)$**: Doubling the sequence length quadruples the memory. We measured this on real hardware.

2. **The quadratic wall is real**: Beyond ~32K tokens (for 32 heads), the attention matrix alone exceeds the memory of any single GPU.

3. **Online softmax enables chunked attention**: We can split K, V into arbitrary chunks, process them one at a time with online softmax, and get **numerically identical** results to full attention.

4. **Chunked attention is the foundation of Ring Attention**: The same algorithm, distributed across GPUs with KV chunks rotating around a ring.

In the next notebook, we will implement Ring Attention itself -- simulating multi-GPU communication on a single device.

In [ ]:
print("=" * 60)
print("SUMMARY: Attention Memory Scaling")
print("=" * 60)
print()
print("Key results:")
print("  - Attention memory = O(S^2) -- verified empirically")
print("  - At 128K tokens, 32 heads: ~1 TB for attention alone")
print("  - Online softmax: chunk-by-chunk attention = exact full attention")
print("  - Context parallelism: per-GPU memory = O(S^2 / N^2)")
print()
print("Next: Implement Ring Attention with simulated multi-GPU communication.")